# 04 · Bliss on the quantile functions: 3-d vectors and the (α, β) plane

Notebook 03 compares the **means**. Here the whole **distribution** of the mothers is used: for the elongation
velocity $v$, the rate $\lambda$ and the length $L$ of each mother in a window (the same per-mother values as in
notebook 03), each condition is described by its **quantile function** $Q_c(u)$, the value of the mother at rank $u$
(from the lowest, $u \approx 0$, to the highest, $u \approx 1$).

Given a control, two monotherapies A and B and their combination, for a window of time you choose:

1. **Bliss on every rank**: $Q_{\rm Bliss}(u) = Q_{\rm ctrl}(u)\,\rho_A(u)\,\rho_B(u)$, with $\rho_c = Q_c/Q_{\rm ctrl}$;
2. **the effects as vectors in 3-d**: where the observed combination is with respect to each drug alone, their sum
   and Bliss;
3. **how much of each drug is in the combination**: the plane $Q_{AB} \approx Q_{\rm ctrl}\,\rho_A^{\alpha}\rho_B^{\beta}$,
   with a bootstrap cloud of $(\alpha, \beta)$: $(1,1)$ is Bliss, $(0,1)$ is "the combination is B alone",
   $(1,0)$ "A alone".

### How to use it

1. Fill in the cell **Your data** below: one line per folder of Excel files (`*_longitud_madres.xlsx`), with the
   areas you want and the condition they belong to.
2. Run all the cells (**Run › Run All Cells**).
3. The figures (PNG, 200 dpi, sized for slides), the tables (CSV) and the settings (`settings.json`) are saved in
   `results/04_quantile_bliss/<RUN_NAME>/`.

Nothing else has to be changed. The cells after **Your data** can be read but do not need to be edited.
Everything is done by functions of `mmtools` (the same ones a graphical interface would call:
`mm.run_quantile_bliss(EXPERIMENTS, ...)` does the whole notebook in one line).

## Your data

**`EXPERIMENTS`**: one `dict` per folder and condition.

| key | what to write | example |
|---|---|---|
| `folder` | folder with the `<area>-<date>_longitud_madres.xlsx` files (absolute, or relative to this notebook) | `'../November/6/Stacks/Excel'` |
| `condition` | name of the condition; several lines can share one condition (e.g. two experiments of the same treatment) | `'NIT'` |
| `areas` | `'all'`, a range `'1-10'`, or a list `'1-5, 8, 12-14'` | `'11-20'` |
| `flow_start` | time the flow started (`'HH:MM'`), which becomes time 0. If you leave it out (or `None`), time 0 is the **first image of that folder** | `'10:17'` |
| `name` | *(optional)* label of the experiment; by default the date in the file names | `'06/11'` |

**Other settings**

- `ROLES`, `WINDOW`, `T_END`, `UM_PER_PX`, `COVER`, `DT_RANGE`: as in notebook 03.
- `VARIABLES`: which of `'v'` (velocity), `'lam'` (rate), `'L'` (length) to analyse.
- `N_LEVELS`: number of quantile levels $u_k = (k - \tfrac12)/N$.
- `V_MIN`: levels where the control, A or B is below this (in units of the control median) are left out of the
  fit (the ratios $\rho$ need positive values: e.g. mothers that are not growing).
- `MIN_EFFECT`: if the effect of a drug alone (RMS of $Q_{\rm drug} - Q_{\rm ctrl}$, control-median units) is below
  this, its exponent cannot be estimated and is fixed at 1 in the main estimate (the fit with both exponents free
  is always reported too).
- `N_BOOT`: bootstrap replicates.
- `OTHER_WINDOWS`: other windows for a sensitivity table (`[]` to skip).

In [ ]:
EXPERIMENTS = [
    dict(folder='../Analisis_NIT_GEN/data_gen_nit_bliss/LB_16_10_excel', areas='all',   condition='LB',      flow_start='10:45'),
    dict(folder='../November/6/Stacks/Excel',                         areas='31-40', condition='GEN',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='11-30', condition='NIT',     flow_start='10:17'),
    dict(folder='../November/6/Stacks/Excel',                         areas='1-10',  condition='NIT+GEN', flow_start='10:17'),
]

ROLES = dict(control='LB', A='GEN', B='NIT', combination='NIT+GEN')
WINDOW = (125, 170)     # min since time 0
T_END = 170             # min, or None
VARIABLES = ['v', 'lam', 'L']
OTHER_WINDOWS = [(80, 125), (95, 170), (140, 170)]

RUN_NAME = 'NIT_GEN_125_170min'
UM_PER_PX = None        # e.g. 0.065; None = pixels
COVER = 0.6
DT_RANGE = (3, 25)      # min
N_LEVELS = 100
V_MIN = 0.05
MIN_EFFECT = 0.1
N_BOOT = 500
CONDITION_COLORS = {}   # optional, e.g. {'LB': '#52514e'}

## 1. Reading the Excel files

First the settings are checked (folders, areas, times); if something cannot be read, the notebook stops here and
says what. Then, for each folder, every `*_longitud_madres.xlsx` of the chosen areas is read: the time of each
image (row 1) and, for each channel, the mother's length and, in the row below, the daughter's length in the
image where a division is first seen.

In [ ]:
NOTEBOOK = '04_quantile_bliss'
SETTINGS = dict(roles=ROLES, window=list(WINDOW), t_end=T_END, variables=VARIABLES, other_windows=OTHER_WINDOWS,
                um_per_px=UM_PER_PX, cover=COVER, dt_range=list(DT_RANGE), n_levels=N_LEVELS, v_min=V_MIN,
                min_effect=MIN_EFFECT, n_boot=N_BOOT)

import warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import mmtools as mm
from mmtools import plots as mp
warnings.filterwarnings('ignore')
mp.set_style()

problems = mm.check_entries(EXPERIMENTS)
if problems:
    raise ValueError('Please fix the cell "Your data":\n  - ' + '\n  - '.join(problems))

ds = mm.load_dataset(EXPERIMENTS, step=1, t_end=T_END, condition_colors=CONDITION_COLORS)
print('\n'.join(ds.messages))
OUT = Path('results') / NOTEBOOK / RUN_NAME
res = mm.AnalysisResult(NOTEBOOK, SETTINGS, ds.experiments, messages=list(ds.messages))

def show(name, fig):
    # show a figure and keep it to be saved at the end
    res.figures[name] = fig
    plt.show()

ds.summary

## 2. One value per mother, and the quantile functions

Each mother gets its time-weighted velocity, rate and length over its intervals in the window, exactly as in
notebook 03 (section 2 there). For each variable and condition, the quantile function is computed on `N_LEVELS`
levels and divided by the **median of the control**, so that 1 is a typical control mother.

The bootstrap (here and below) resamples the **areas** of each condition with replacement, all four conditions
at once, and recomputes everything (including the control median used as unit). This takes a little while.

In [ ]:
S = mm.quantile_bliss_analysis(ds, ROLES, WINDOW, variables=VARIABLES, cover=COVER, dt_range=DT_RANGE,
                               um_per_px=UM_PER_PX, n_levels=N_LEVELS, v_min=V_MIN, min_effect=MIN_EFFECT, n_boot=N_BOOT)
S.W.groupby('condition').size().reindex(list(ROLES.values())).rename('mothers in the window').to_frame()

## 3. Bliss on every rank

**Left**, one row per variable: the quantile functions of the four conditions, the Bliss prediction
$Q_{\rm ctrl}\rho_A\rho_B$ (dotted) and the fit of the plane of section 5 (dashed).
**Right**: the log-effects at each rank, $\ln\rho_c(u)$. In log scale Bliss is just the sum of the two drugs
(dotted). Where the observed curve follows one drug and not the sum, the combination behaves like that drug.

A rank $u$ is not a mother: the mother at rank 0.3 of the combination is not "the same" as the mother at rank 0.3
of the control. Bliss on quantiles assumes that each drug acts on the rank (the slowest mothers stay the slowest),
which is the natural extension of Bliss to distributions but not a test of it.

In [ ]:
show('quantile_functions', S.fig_quantiles())

## 4. The effects as vectors in 3-d

**Vectors.** The linear effects $e_c(u) = Q_c(u) - Q_{\rm ctrl}(u)$ (control-median units) on the levels used in the
fit, with the inner product $\langle v, w\rangle = \frac1N\sum_u v(u)\,w(u)$ and its norm. Each effect is one vector;
angles and distances between them say how similar two effects are, rank by rank.

**Why three dimensions.** In linear units Bliss is not the sum of the two effects:
$e_{\rm Bliss} = e_A + e_B + e_A e_B / Q_{\rm ctrl}$. The cross term is a product rank by rank, so it can stick out
of the plane of $e_A$ and $e_B$. The figure uses the 3-d space spanned by $e_A$, $e_B$ and $e_{\rm Bliss}$, with an
orthonormal basis (Gram–Schmidt): $u_1$ along $e_A$, $u_2$ completing the plane of $e_A$ and $e_B$, $u_3$ the part of
$e_{\rm Bliss}$ outside that plane. $e_A$, $e_B$, their plain sum and $e_{\rm Bliss}$ are drawn exactly; the observed
effect (and the plane fit) can also have a part outside this space: the arrow is its projection, and the length of
the rest is in the title.

**Left**: arrows from the origin (the control): the two drugs, their plain sum (grey dashed, corner of the dotted
parallelogram), Bliss (black), the plane fit (pink dashed) and the observed combination. The black dotted segment
from the sum to Bliss is the cross term; the red segment from Bliss to the observed effect is the deviation from
Bliss. Grey drop lines: height above the plane. **Right**: the same vectors along the rank.

How to read it: if the observed arrow ends near Bliss, Bliss describes the combination; if it ends near one drug,
the combination behaves like that drug alone.

In [ ]:
for v in VARIABLES:
    show(f'vectors_3d_{v}', S.fig_vectors(v))

Angles (degrees) and distances (control-median units) between the vectors:

In [ ]:
S.vectors_table().round(3).T

## 5. How much of each drug is in the combination: the (α, β) plane

The simplest model that keeps the Bliss form but lets each drug act with its own strength:

$$Q_{AB}(u) \;\approx\; Q_{\rm ctrl}(u)\,\rho_A(u)^{\alpha}\,\rho_B(u)^{\beta}
\qquad\Longleftrightarrow\qquad \ln\rho_{AB} \approx \alpha\ln\rho_A + \beta\ln\rho_B .$$

| $(\alpha,\beta)$ | meaning |
|---|---|
| $(1,1)$ | Bliss: both drugs act at full strength, independently |
| $(0,1)$ | the combination behaves like B alone |
| $(1,0)$ | the combination behaves like A alone |
| $(0,0)$ | no effect (the control) |
| $\alpha>1$ or $\beta>1$ | that drug is enhanced by the other |

**Fit.** $\alpha, \beta$ minimise $\frac1N\sum_u\big[Q_{\rm ctrl}\rho_A^{\alpha}\rho_B^{\beta} - Q_{AB}\big]^2$ in the
units of the variable (not in logarithms, so that mothers at 0 in the combination can enter), on the levels where
the control, A and B are all ≥ `V_MIN`, with $\alpha,\beta\in[-3,5]$ starting at $(1,1)$.

**A drug that cannot be estimated.** If a drug alone has almost no effect ($\|e\| <$ `MIN_EFFECT`), $\rho\approx 1$
and its exponent does not change the fit: it is fixed at 1 in the main estimate (decided once on the data and kept
in every resample). The columns *both free* give the fit with both exponents estimated anyway.

**Uncertainty.** From the bootstrap pairs $(\alpha,\beta)$: 95 % intervals of each exponent, their correlation, the
95 % ellipse of the cloud, and the squared Mahalanobis distance $D^2$ of each reference point to it (inside the
ellipse if $D^2$ is below the threshold: 5.99 with two estimated exponents, 3.84 with one). The last columns are
the RMS distance (control-median units) of the observed quantile function to each model.

**Figure**: the bootstrap cloud with both exponents free (grey), its 95 % ellipse, the estimate and the reference
points. When the two effects have opposite signs the cloud is a long diagonal: the data fix a combination of the
exponents better than each one (more of A has to be compensated by more of B).

In [ ]:
show('alpha_beta', S.fig_alpha_beta())
S.table().T

## 6. Other windows

The same $(\alpha,\beta)$ table for the windows in `OTHER_WINDOWS` (300 resamples each), to see whether the
conclusion depends on the window. In a window where one drug has not acted yet, its exponent says nothing.

In [ ]:
if OTHER_WINDOWS:
    SENS = mm.quantile_bliss_windows(ds, ROLES, [WINDOW] + list(OTHER_WINDOWS), n_boot=300, variables=VARIABLES,
                                     cover=COVER, dt_range=DT_RANGE, um_per_px=UM_PER_PX, n_levels=N_LEVELS,
                                     v_min=V_MIN, min_effect=MIN_EFFECT)
    res.tables['alpha_beta_windows'] = SENS
    display(SENS[['mothers', f"α ({ROLES['A']})", f"β ({ROLES['B']})", 'α, both free', 'β, both free',
                  'D² Bliss (1,1)', f"D² {ROLES['B']} alone (0,1)", f"D² {ROLES['A']} alone (1,0)"]].round(2))

## 7. Saving

In `results/04_quantile_bliss/<RUN_NAME>/`: the figures (numbered PNG) and

- `alpha_beta.csv`: section 5; `alpha_beta_windows.csv`: section 6;
- `vectors_3d.csv`: angles and distances of section 4;
- `quantile_functions.csv`: the quantile functions of every condition and the Bliss prediction, in the units of
  the variable;
- `experiments.csv`, `settings.json` (`mm.load_config` reads it back), `messages.txt`.

In [ ]:
res.tables.update(S.tables())
res.tables['experiments'] = ds.summary
res.save(OUT)
print('saved in', OUT.resolve())
sorted(p.name for p in OUT.iterdir())